# 00 — Reto 1: Data Prep

Notebook de preparación de datos del **Reto 1 — Operational Intelligence**. Es la puerta de entrada al trabajo de datos del reto. Debe ejecutarse antes que cualquier otro notebook de la secuencia.

**Qué produce:**
- `data/interim/metrics_raw_cleaned.parquet` — métricas limpias (wide, pre-melt)
- `data/interim/orders_raw_cleaned.parquet` — órdenes limpias (wide, pre-melt)
- `data/processed/metrics_long.parquet` — métricas en formato long (grano: país/ciudad/zona/métrica/semana)
- `data/processed/orders_long.parquet` — órdenes en formato long
- `data/processed/zone_master.parquet` — cobertura de zonas entre fuentes

**Qué no hace:**
- No imputa faltantes.
- No colapsa duplicados lógicos ambiguos.
- No asigna fechas calendario (solo offsets relativos L8W–L0W).
- No infiere semántica de negocio sobre métricas.

**Diseño:** proceso visible y auditable dentro del notebook. Los helpers de `src/helpers/` solo resuelven paths y I/O mecánico — la lógica de negocio del pipeline está aquí.

**Secuencia del Reto 1:**
```
00_reto1_data_prep.ipynb   ← este notebook
10_reto1_eda.ipynb
20_reto1_semantic_layer.ipynb
30_reto1_insight_engine.ipynb
40_reto1_chatbot_design.ipynb
```

## 0) Setup

In [1]:
from __future__ import annotations

import json
import re
import sys
import warnings
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

# Add src to path so helpers are importable from notebook
ROOT = Path.cwd()
for p in [ROOT] + list(ROOT.parents):
    if (p / 'data' / 'processed').exists() and (p / 'notebooks').exists():
        ROOT = p
        break

if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from helpers.paths import get_paths, default_excel
from helpers.io import load_sheet, detect_week_columns, coerce_numeric, normalize_text, write_parquet

PATHS = get_paths(ROOT)
print('ROOT          :', ROOT)
print('raw_dir       :', PATHS.raw_dir)
print('interim_dir   :', PATHS.interim_dir)
print('processed_dir :', PATHS.processed_dir)
print('reports_dir   :', PATHS.reports_dir)

ROOT          : /home/thechieft/projects/IAeng
raw_dir       : /home/thechieft/projects/IAeng/data/raw
interim_dir   : /home/thechieft/projects/IAeng/data/interim
processed_dir : /home/thechieft/projects/IAeng/data/processed
reports_dir   : /home/thechieft/projects/IAeng/reports


## 1) Configuración

Parámetros configurables del pipeline. Cambiar `RETO` para adaptar a otro contexto futuro.

In [2]:
RETO = 1  # 1 = Operational Intelligence | 2 = Competitive Intelligence (datos distintos)

EXCEL_PATH = default_excel(PATHS)
print(f'RETO         : {RETO}')
print(f'Excel fuente : {EXCEL_PATH.name}')
print(f'Existe       : {EXCEL_PATH.exists()}')

# Sheets esperadas para Reto 1
EXPECTED_SHEETS = ['RAW_INPUT_METRICS', 'RAW_ORDERS', 'RAW_SUMMARY']

RETO         : 1
Excel fuente : Sistema de Análisis Inteligente para Operaciones Rappi - Dummy Data (1).xlsx
Existe       : True


## 2) Carga y perfilado raw

**Qué se hace:** Carga de las tres hojas del workbook y generación de un perfil básico de estructura.

**Por qué perfilar primero:** Antes de cualquier transformación, se documenta el estado crudo del dato. Esto permite comparar contra el output limpio y detectar pérdidas o transformaciones no esperadas.

**Shapes esperados (observados en datos reales — no en el PDF del caso):**
- `RAW_INPUT_METRICS`: 12,573 × 15 (incluye columnas semanales con sufijo `_ROLL`)
- `RAW_ORDERS`: 1,242 × 13 (columnas semanales sin sufijo)
- `RAW_SUMMARY`: 15 × 4 (tabla de referencia, no se transforma)

In [3]:
raw_metrics = load_sheet(EXCEL_PATH, 'RAW_INPUT_METRICS')
raw_orders  = load_sheet(EXCEL_PATH, 'RAW_ORDERS')
raw_summary = load_sheet(EXCEL_PATH, 'RAW_SUMMARY')

raw_profile = pd.DataFrame([
    {
        'sheet': name,
        'rows': df.shape[0],
        'cols': df.shape[1],
        'blank_rows': int(df.isna().all(axis=1).sum()),
        'exact_duplicates': int(df.duplicated().sum()),
    }
    for name, df in [('RAW_INPUT_METRICS', raw_metrics), ('RAW_ORDERS', raw_orders), ('RAW_SUMMARY', raw_summary)]
])

display(raw_profile)

# Guardar schema crudo como referencia
schema_out = PATHS.reports_dir / 'reto1' / 'raw_schema_summary.json'
schema_out.parent.mkdir(parents=True, exist_ok=True)
schema_summary = {
    'excel_path': str(EXCEL_PATH),
    'sheets': {
        row['sheet']: {
            'shape': [row['rows'], row['cols']],
            'blank_rows': row['blank_rows'],
            'exact_duplicates': row['exact_duplicates'],
        }
        for _, row in raw_profile.iterrows()
    }
}
schema_out.write_text(json.dumps(schema_summary, indent=2), encoding='utf-8')
print(f'Schema guardado: {schema_out}')

,sheet,rows,cols,blank_rows,exact_duplicates
0,RAW_INPUT_METRICS,12573,15,0,963
1,RAW_ORDERS,1242,13,0,0
2,RAW_SUMMARY,15,4,0,0


Schema guardado: /home/thechieft/projects/IAeng/reports/reto1/raw_schema_summary.json


## 3) Limpieza — Métricas

**Hoja:** `RAW_INPUT_METRICS`  
**Output:** `data/interim/metrics_raw_cleaned.parquet`

### Decisiones de limpieza

**3.1 Columnas fuente preservadas para dedup exacto**  
Se capturan los nombres de columna originales *antes* de agregar columnas auxiliares (`_SOURCE_ROW_NUMBER`). El dedup exacto opera sobre columnas fuente para no verse contaminado por la numeración auxiliar.

**3.2 Deduplicación conservadora**  
Solo se eliminan filas *exactamente* idénticas en todas las columnas fuente. Filas que comparten key lógica pero difieren en algún atributo no se tocan — se reportan para revisión posterior pero no se colapsan automáticamente.

**3.3 Normalización de texto**  
- `COUNTRY`: trim + uppercase (es código de país, semántica mayúscula tiene sentido).
- `CITY`, `ZONE`, `ZONE_TYPE`, `ZONE_PRIORITIZATION`, `METRIC`: trim + normalización de espacios internos múltiples. **No** se renombran ni estandarizan manualmente — la semántica geográfica y de métrica se preserva tal como está en el dato.
- Las columnas originales se conservan con sufijo `_ORIGINAL` para trazabilidad.

**3.4 Mapeo canónico de columnas semanales**  
Las columnas semanales en `RAW_INPUT_METRICS` tienen sufijo `_ROLL` (ej: `L8W_ROLL`). Se mapean a canónico `L8W`–`L0W` mediante regex. Las columnas originales se conservan para trazabilidad. Este mapeo es extensible a columnas tipo `_VALUE` si aparecen en versiones futuras del archivo.

**3.5 Coerción numérica con trazabilidad**  
Las semanas canónicas se convierten a numérico con `pd.to_numeric(errors='coerce')`. Se registran fallas por columna. Resultado esperado: 0 fallas (confirmado en datos actuales).

In [4]:
TEXT_DIM_METRICS = ['COUNTRY', 'CITY', 'ZONE', 'ZONE_TYPE', 'ZONE_PRIORITIZATION', 'METRIC']

def clean_sheet(
    df: pd.DataFrame,
    text_dims: List[str],
    source_table_name: str,
) -> Tuple[pd.DataFrame, dict]:
    """Limpieza conservadora: blank rows, exact dedup, text normalize, week coerce."""
    source_columns = list(df.columns)  # capturar antes de agregar auxiliares
    profile = {
        'source_table': source_table_name,
        'raw_rows': int(df.shape[0]),
        'blank_rows_removed': 0,
        'exact_duplicates_removed': 0,
        'week_column_mapping': {},
        'numeric_coercion_failures': {},
    }

    df = df.copy()
    df['_SOURCE_ROW_NUMBER'] = df.index + 2  # +2 = encabezado Excel

    # Blank rows
    blank_mask = df[source_columns].isna().all(axis=1)
    profile['blank_rows_removed'] = int(blank_mask.sum())
    df = df.loc[~blank_mask].copy()

    # Exact dedup (sobre columnas fuente, no auxiliares)
    dup_mask = df.duplicated(subset=source_columns, keep='first')
    profile['exact_duplicates_removed'] = int(dup_mask.sum())
    df = df.loc[~dup_mask].copy()

    # Text normalization
    for col in text_dims:
        if col not in df.columns:
            continue
        df[f'{col}_ORIGINAL'] = df[col]
        df[col] = df[col].map(lambda x: normalize_text(x, upper=(col == 'COUNTRY')))

    # Week columns: detect, add canonical, coerce
    week_mapping = detect_week_columns(df.columns)
    profile['week_column_mapping'] = dict(sorted(week_mapping.items()))

    for src_col, canon_col in week_mapping.items():
        numeric_series, failures = coerce_numeric(df[src_col])
        df[canon_col] = numeric_series
        profile['numeric_coercion_failures'][canon_col] = (
            profile['numeric_coercion_failures'].get(canon_col, 0) + failures
        )

    df['SOURCE_TABLE'] = source_table_name
    profile['clean_rows'] = int(df.shape[0])
    return df, profile


metrics_clean, metrics_profile = clean_sheet(raw_metrics, TEXT_DIM_METRICS, 'RAW_INPUT_METRICS')

print('--- Métricas: perfil de limpieza ---')
for k, v in metrics_profile.items():
    if k not in ('week_column_mapping', 'numeric_coercion_failures'):
        print(f'  {k}: {v}')
print(f'  week_col_mapping ({len(metrics_profile["week_column_mapping"])} cols):', metrics_profile['week_column_mapping'])
print(f'  coercion_failures:', metrics_profile['numeric_coercion_failures'])

--- Métricas: perfil de limpieza ---
  source_table: RAW_INPUT_METRICS
  raw_rows: 12573
  blank_rows_removed: 0
  exact_duplicates_removed: 963
  clean_rows: 11610
  week_col_mapping (9 cols): {'L0W_ROLL': 'L0W', 'L1W_ROLL': 'L1W', 'L2W_ROLL': 'L2W', 'L3W_ROLL': 'L3W', 'L4W_ROLL': 'L4W', 'L5W_ROLL': 'L5W', 'L6W_ROLL': 'L6W', 'L7W_ROLL': 'L7W', 'L8W_ROLL': 'L8W'}
  coercion_failures: {'L8W': 0, 'L7W': 0, 'L6W': 0, 'L5W': 0, 'L4W': 0, 'L3W': 0, 'L2W': 0, 'L1W': 0, 'L0W': 0}


In [5]:
# Reorder columns: dims -> source info -> original cols -> original week cols -> canonical week cols
def reorder_metrics_cols(df: pd.DataFrame) -> pd.DataFrame:
    dims = ['COUNTRY', 'CITY', 'ZONE', 'ZONE_TYPE', 'ZONE_PRIORITIZATION', 'METRIC', 'SOURCE_TABLE', '_SOURCE_ROW_NUMBER']
    originals = [c for c in df.columns if c.endswith('_ORIGINAL')]
    orig_week  = sorted([c for c in df.columns if (c.endswith('_ROLL') or c.endswith('_VALUE'))])
    canon_week = [f'L{i}W' for i in range(8, -1, -1) if f'L{i}W' in df.columns]
    ordered = dims + originals + orig_week + canon_week
    remainder = [c for c in df.columns if c not in ordered]
    return df[ordered + remainder]

metrics_clean = reorder_metrics_cols(metrics_clean)

out = write_parquet(metrics_clean, PATHS.interim_dir / 'metrics_raw_cleaned.parquet')
print(f'Guardado: {out}  shape={metrics_clean.shape}')
display(metrics_clean.head(3))

Guardado: /home/thechieft/projects/IAeng/data/interim/metrics_raw_cleaned.parquet  shape=(11610, 32)


,COUNTRY,CITY,ZONE,ZONE_TYPE,ZONE_PRIORITIZATION,METRIC,SOURCE_TABLE,_SOURCE_ROW_NUMBER,COUNTRY_ORIGINAL,CITY_ORIGINAL,ZONE_ORIGINAL,ZONE_TYPE_ORIGINAL,ZONE_PRIORITIZATION_ORIGINAL,METRIC_ORIGINAL,L0W_ROLL,L1W_ROLL,L2W_ROLL,L3W_ROLL,L4W_ROLL,L5W_ROLL,L6W_ROLL,L7W_ROLL,L8W_ROLL,L8W,L7W,L6W,L5W,L4W,L3W,L2W,L1W,L0W
0,EC,Quito,San Martin de Porras,Non Wealthy,Not Prioritized,Retail SST > SS CVR,RAW_INPUT_METRICS,2,EC,Quito,San Martin de Porras,Non Wealthy,Not Prioritized,Retail SST > SS CVR,0.735549,0.670137,0.676429,0.731758,0.781532,0.709402,0.657760,0.538596,0.602339,0.602339,0.538596,0.657760,0.709402,0.781532,0.731758,0.676429,0.670137,0.735549
1,BR,Bauru,FS Parque Jardim Europa/Vila Riachuelo - BAU,Non Wealthy,Not Prioritized,Restaurants SST > SS CVR,RAW_INPUT_METRICS,3,BR,Bauru,FS Parque Jardim Europa/Vila Riachuelo - BAU,Non Wealthy,Not Prioritized,Restaurants SST > SS CVR,0.475000,0.450000,0.250000,0.425000,0.450000,0.525000,0.880000,0.926667,0.857544,0.857544,0.926667,0.880000,0.525000,0.450000,0.425000,0.250000,0.450000,0.475000
2,MX,Mexicali,MXL_Universidad,Wealthy,Prioritized,Retail SST > SS CVR,RAW_INPUT_METRICS,4,MX,Mexicali,MXL_Universidad,Wealthy,Prioritized,Retail SST > SS CVR,0.897864,0.897732,0.894802,0.888999,0.895019,0.900928,0.905766,0.904237,0.900823,0.900823,0.904237,0.905766,0.900928,0.895019,0.888999,0.894802,0.897732,0.897864


## 4) Limpieza — Órdenes

**Hoja:** `RAW_ORDERS`  
**Output:** `data/interim/orders_raw_cleaned.parquet`

Misma lógica que métricas. Diferencias:
- Dimensiones de texto: `COUNTRY`, `CITY`, `ZONE`, `METRIC` (sin `ZONE_TYPE` ni `ZONE_PRIORITIZATION`).
- Columnas semanales sin sufijo (ya son canónicas `L8W`–`L0W` directamente).
- **Resultado de dedup:** 0 duplicados exactos (confirmado — a diferencia de métricas que tiene 963).

In [6]:
TEXT_DIM_ORDERS = ['COUNTRY', 'CITY', 'ZONE', 'METRIC']

orders_clean, orders_profile = clean_sheet(raw_orders, TEXT_DIM_ORDERS, 'RAW_ORDERS')

def reorder_orders_cols(df: pd.DataFrame) -> pd.DataFrame:
    dims = ['COUNTRY', 'CITY', 'ZONE', 'METRIC', 'SOURCE_TABLE', '_SOURCE_ROW_NUMBER']
    originals = [c for c in df.columns if c.endswith('_ORIGINAL')]
    canon_week = [f'L{i}W' for i in range(8, -1, -1) if f'L{i}W' in df.columns]
    ordered = dims + originals + canon_week
    remainder = [c for c in df.columns if c not in ordered]
    return df[ordered + remainder]

orders_clean = reorder_orders_cols(orders_clean)

out = write_parquet(orders_clean, PATHS.interim_dir / 'orders_raw_cleaned.parquet')
print(f'Guardado: {out}  shape={orders_clean.shape}')

print('\n--- Órdenes: perfil de limpieza ---')
for k, v in orders_profile.items():
    if k not in ('week_column_mapping', 'numeric_coercion_failures'):
        print(f'  {k}: {v}')
print(f'  coercion_failures:', orders_profile['numeric_coercion_failures'])
display(orders_clean.head(3))

Guardado: /home/thechieft/projects/IAeng/data/interim/orders_raw_cleaned.parquet  shape=(1242, 19)

--- Órdenes: perfil de limpieza ---
  source_table: RAW_ORDERS
  raw_rows: 1242
  blank_rows_removed: 0
  exact_duplicates_removed: 0
  clean_rows: 1242
  coercion_failures: {'L8W': 0, 'L7W': 0, 'L6W': 0, 'L5W': 0, 'L4W': 0, 'L3W': 0, 'L2W': 0, 'L1W': 0, 'L0W': 0}


,COUNTRY,CITY,ZONE,METRIC,SOURCE_TABLE,_SOURCE_ROW_NUMBER,COUNTRY_ORIGINAL,CITY_ORIGINAL,ZONE_ORIGINAL,METRIC_ORIGINAL,L8W,L7W,L6W,L5W,L4W,L3W,L2W,L1W,L0W
0,CO,Bogota,Chapinero,Orders,RAW_ORDERS,2,CO,Bogota,Chapinero,Orders,264650.0,261767.0,277005.0,279886.0,277954.0,275344.0,275633.0,279048.0,274376.0
1,CO,Bogota,Usaquén,Orders,RAW_ORDERS,3,CO,Bogota,Usaquén,Orders,233674.0,224436.0,241602.0,240411.0,240786.0,236591.0,241475.0,238649.0,236876.0
2,PE,Lima,Miraflores,Orders,RAW_ORDERS,4,PE,Lima,Miraflores,Orders,123252.0,125820.0,126245.0,123770.0,126467.0,124376.0,124751.0,123431.0,124757.0


## 5) Transformación wide → long

**Qué se hace:** Las tablas limpias (wide) se transforman al formato long, donde cada fila es una entidad × semana.

**Grano resultante:** `(COUNTRY, CITY, ZONE, METRIC, WEEK_OFFSET)` — debe ser único. Validado en sección 7.

**Columnas generadas:**
- `WEEK_OFFSET`: etiqueta canónica `L8W`–`L0W`.
- `week_offset_num`: entero 0–8 (0 = semana actual, 8 = 8 semanas atrás).
- `is_current_week`: booleano, `True` solo para `L0W`.
- `has_missing_history`: `True` si la entidad tiene al menos un `VALUE` nulo en cualquier semana.
- `metric_group`: grupo estructural inferido del nombre de métrica (antes del símbolo `>` o primer token). Es indicativo, no catálogo oficial.

**Matemática esperada:** `clean_rows × 9 semanas = long_rows`. Verificado en validación.

In [7]:
def week_offset_num(week_offset: str) -> int:
    m = re.match(r'^L([0-8])W$', str(week_offset).upper())
    if not m:
        raise ValueError(f'Invalid week offset: {week_offset}')
    return int(m.group(1))


def infer_metric_group(metric: object) -> object:
    if pd.isna(metric):
        return pd.NA
    text = str(metric)
    return text.split('>', 1)[0].strip() if '>' in text else text.split(' ', 1)[0].strip()


def make_long(
    df: pd.DataFrame,
    id_vars: List[str],
    source_name: str,
) -> pd.DataFrame:
    week_cols = sorted(
        [c for c in df.columns if re.match(r'^L[0-8]W$', c)],
        key=lambda c: int(c[1]),
        reverse=True,
    )
    long_df = df.melt(
        id_vars=[c for c in id_vars if c in df.columns],
        value_vars=week_cols,
        var_name='WEEK_OFFSET',
        value_name='VALUE',
    )
    long_df['week_offset_num'] = long_df['WEEK_OFFSET'].map(week_offset_num)
    long_df['is_current_week']  = long_df['week_offset_num'].eq(0)

    hist_key = [c for c in ['COUNTRY', 'CITY', 'ZONE', 'METRIC'] if c in long_df.columns]
    long_df['has_missing_history'] = long_df.groupby(hist_key)['VALUE'].transform(lambda s: s.isna().any())
    return long_df


# --- Metrics long ---
metrics_id_vars = ['COUNTRY', 'CITY', 'ZONE', 'ZONE_TYPE', 'ZONE_PRIORITIZATION',
                   'METRIC', 'SOURCE_TABLE', '_SOURCE_ROW_NUMBER']

metrics_long = make_long(metrics_clean, metrics_id_vars, 'RAW_INPUT_METRICS')
metrics_long['metric_group'] = metrics_long['METRIC'].map(infer_metric_group)

metrics_long = metrics_long[[
    'COUNTRY', 'CITY', 'ZONE', 'ZONE_TYPE', 'ZONE_PRIORITIZATION',
    'METRIC', 'metric_group', 'WEEK_OFFSET', 'week_offset_num',
    'VALUE', 'SOURCE_TABLE', 'is_current_week', 'has_missing_history', '_SOURCE_ROW_NUMBER',
]]

# --- Orders long ---
orders_id_vars = ['COUNTRY', 'CITY', 'ZONE', 'METRIC', 'SOURCE_TABLE', '_SOURCE_ROW_NUMBER']

orders_long = make_long(orders_clean, orders_id_vars, 'RAW_ORDERS')
orders_long = orders_long[[
    'COUNTRY', 'CITY', 'ZONE', 'METRIC', 'WEEK_OFFSET', 'week_offset_num',
    'VALUE', 'SOURCE_TABLE', 'is_current_week', 'has_missing_history', '_SOURCE_ROW_NUMBER',
]]

# Export
write_parquet(metrics_long, PATHS.processed_dir / 'metrics_long.parquet')
write_parquet(orders_long,  PATHS.processed_dir / 'orders_long.parquet')

print('metrics_long:', metrics_long.shape)
print('orders_long :', orders_long.shape)
print(f'Matemática métricas: {metrics_clean.shape[0]} × 9 = {metrics_clean.shape[0]*9} → {metrics_long.shape[0]} ✓')
print(f'Matemática órdenes : {orders_clean.shape[0]} × 9 = {orders_clean.shape[0]*9} → {orders_long.shape[0]} ✓')
display(metrics_long.head(3))

metrics_long: (104490, 14)
orders_long : (11178, 11)
Matemática métricas: 11610 × 9 = 104490 → 104490 ✓
Matemática órdenes : 1242 × 9 = 11178 → 11178 ✓


,COUNTRY,CITY,ZONE,ZONE_TYPE,ZONE_PRIORITIZATION,METRIC,metric_group,WEEK_OFFSET,week_offset_num,VALUE,SOURCE_TABLE,is_current_week,has_missing_history,_SOURCE_ROW_NUMBER
0,EC,Quito,San Martin de Porras,Non Wealthy,Not Prioritized,Retail SST > SS CVR,Retail SST,L8W,8,0.602339,RAW_INPUT_METRICS,False,False,2
1,BR,Bauru,FS Parque Jardim Europa/Vila Riachuelo - BAU,Non Wealthy,Not Prioritized,Restaurants SST > SS CVR,Restaurants SST,L8W,8,0.857544,RAW_INPUT_METRICS,False,False,3
2,MX,Mexicali,MXL_Universidad,Wealthy,Prioritized,Retail SST > SS CVR,Retail SST,L8W,8,0.900823,RAW_INPUT_METRICS,False,False,4


## 6) Zone master

**Qué es:** Tabla de cobertura de zonas. Registra qué zonas existen en cada fuente (métricas, órdenes, o ambas).

**Por qué existe:** Las zonas en `RAW_INPUT_METRICS` y `RAW_ORDERS` no son idénticas. Hacer un join inner silenciosamente excluiría 264 zonas. El `zone_master` hace ese mismatch explícito y auditables las decisiones de join posteriores.

**`COVERAGE_CLASS`:**
- `BOTH`: zona presente en métricas y órdenes.
- `ONLY_METRICS`: zona en métricas, no en órdenes (2 zonas).
- `ONLY_ORDERS`: zona en órdenes, no en métricas (264 zonas).

In [8]:
zone_key = ['COUNTRY', 'CITY', 'ZONE']

metrics_zones = (
    metrics_clean
    .groupby(zone_key, dropna=False)
    .agg(
        METRIC_COUNT=('METRIC', 'nunique'),
        ZONE_TYPE_NUNIQUE=('ZONE_TYPE', 'nunique'),
        ZONE_PRIORITIZATION_NUNIQUE=('ZONE_PRIORITIZATION', 'nunique'),
        ZONE_TYPE=('ZONE_TYPE', 'first'),
        ZONE_PRIORITIZATION=('ZONE_PRIORITIZATION', 'first'),
    )
    .reset_index()
)
metrics_zones['IN_METRICS'] = True

orders_zones = orders_clean[zone_key].drop_duplicates().copy()
orders_zones['IN_ORDERS'] = True

zone_master = metrics_zones.merge(orders_zones, on=zone_key, how='outer')
zone_master['IN_METRICS'] = zone_master['IN_METRICS'].fillna(False)
zone_master['IN_ORDERS']  = zone_master['IN_ORDERS'].fillna(False)
zone_master['METRIC_COUNT'] = zone_master['METRIC_COUNT'].fillna(0).astype(int)
zone_master['ZONE_TYPE_CONFLICT']          = zone_master['ZONE_TYPE_NUNIQUE'].fillna(0).gt(1)
zone_master['ZONE_PRIORITIZATION_CONFLICT'] = zone_master['ZONE_PRIORITIZATION_NUNIQUE'].fillna(0).gt(1)

zone_master['COVERAGE_CLASS'] = 'ONLY_ORDERS'
zone_master.loc[zone_master['IN_METRICS'] & ~zone_master['IN_ORDERS'], 'COVERAGE_CLASS'] = 'ONLY_METRICS'
zone_master.loc[zone_master['IN_METRICS'] &  zone_master['IN_ORDERS'], 'COVERAGE_CLASS'] = 'BOTH'

zone_master = zone_master.sort_values(zone_key).reset_index(drop=True)

write_parquet(zone_master, PATHS.processed_dir / 'zone_master.parquet')

coverage = zone_master['COVERAGE_CLASS'].value_counts()
print('zone_master shape :', zone_master.shape)
print('Cobertura:')
print(coverage.to_string())

zone_master shape : (1244, 13)
Cobertura:
COVERAGE_CLASS
BOTH            978
ONLY_ORDERS     264
ONLY_METRICS      2


## 7) Validación del pipeline

10 checks estructurales para garantizar que el pipeline no introdujo errores silenciosos. Todos deben ser **PASS** antes de usar los artefactos en análisis.

Si algún check falla, el output de esa celda muestra el check fallido y los valores observados vs. esperados.

In [9]:
checks = []

def chk(check_id: str, passed: bool, details: dict) -> None:
    checks.append({'id': check_id, 'status': 'PASS' if passed else 'FAIL', **details})

# 1. Raw shapes match known source
chk('raw_shapes_match_source',
    raw_metrics.shape == (12573, 15) and raw_orders.shape == (1242, 13) and raw_summary.shape == (15, 4),
    {'raw_metrics': raw_metrics.shape, 'raw_orders': raw_orders.shape, 'raw_summary': raw_summary.shape})

# 2. Exact duplicates in RAW_INPUT_METRICS == 963
chk('metrics_exact_dupes_963',
    metrics_profile['exact_duplicates_removed'] == 963,
    {'observed': metrics_profile['exact_duplicates_removed'], 'expected': 963})

# 3. Orders exact duplicates == 0
chk('orders_no_exact_dupes',
    orders_profile['exact_duplicates_removed'] == 0,
    {'observed': orders_profile['exact_duplicates_removed']})

# 4. Week column mapping detected correctly
expected_m = [f'L{i}W_ROLL' for i in range(8, -1, -1)]
expected_o = [f'L{i}W' for i in range(8, -1, -1)]
chk('week_columns_detected',
    all(c in raw_metrics.columns for c in expected_m) and
    all(c in raw_orders.columns  for c in expected_o),
    {'metrics_roll_cols': expected_m, 'orders_week_cols': expected_o})

# 5. No artificial NaNs from numeric coercion
total_coerce_failures = (
    sum(metrics_profile['numeric_coercion_failures'].values()) +
    sum(orders_profile['numeric_coercion_failures'].values())
)
chk('no_artificial_nans_from_coercion',
    total_coerce_failures == 0,
    {'total_coercion_failures': total_coerce_failures})

# 6. Wide→long row math is correct
exp_ml = metrics_clean.shape[0] * 9
exp_ol = orders_clean.shape[0]  * 9
chk('wide_to_long_row_math',
    metrics_long.shape[0] == exp_ml and orders_long.shape[0] == exp_ol,
    {'metrics_expected': exp_ml, 'metrics_actual': metrics_long.shape[0],
     'orders_expected': exp_ol,  'orders_actual': orders_long.shape[0]})

# 7. Grain uniqueness in long tables
grain = ['COUNTRY', 'CITY', 'ZONE', 'METRIC', 'WEEK_OFFSET']
m_grain_dupes = int(metrics_long.duplicated(subset=grain).sum())
o_grain_dupes = int(orders_long.duplicated(subset=grain).sum())
chk('grain_unique_in_long_tables',
    m_grain_dupes == 0 and o_grain_dupes == 0,
    {'metrics_grain_dupes': m_grain_dupes, 'orders_grain_dupes': o_grain_dupes})

# 8. Zone master == union of both zone sets
mz = set(metrics_clean[zone_key].drop_duplicates().itertuples(index=False, name=None))
oz = set(orders_clean[zone_key].drop_duplicates().itertuples(index=False, name=None))
zm = set(zone_master[zone_key].drop_duplicates().itertuples(index=False, name=None))
chk('zone_master_is_union',
    zm == (mz | oz),
    {'zone_master_count': len(zm), 'union_count': len(mz | oz)})

# 9. Coverage mismatch is real (known structural issue)
only_m = len(mz - oz)
only_o = len(oz - mz)
chk('coverage_mismatch_is_real',
    only_m > 0 or only_o > 0,
    {'only_metrics': only_m, 'only_orders': only_o})

# 10. SOURCE_TABLE tags and WEEK_OFFSET values are correct
chk('processed_outputs_coherent',
    set(metrics_long['SOURCE_TABLE'].unique()) == {'RAW_INPUT_METRICS'} and
    set(orders_long['SOURCE_TABLE'].unique())  == {'RAW_ORDERS'} and
    set(metrics_long['WEEK_OFFSET'].unique())  == {f'L{i}W' for i in range(9)} and
    set(orders_long['WEEK_OFFSET'].unique())   == {f'L{i}W' for i in range(9)},
    {'metrics_source': sorted(metrics_long['SOURCE_TABLE'].unique()),
     'orders_source':  sorted(orders_long['SOURCE_TABLE'].unique())})

# Summary
results_df = pd.DataFrame(checks)[['id', 'status']]
display(results_df)
failed = [c for c in checks if c['status'] == 'FAIL']
print(f"\nOverall: {'PASS' if not failed else 'FAIL'} ({len(checks)-len(failed)}/{len(checks)} passed)")
if failed:
    print('\nFailed checks:')
    for c in failed:
        print(f'  {c}')

,id,status
0,raw_shapes_match_source,PASS
1,metrics_exact_dupes_963,PASS
2,orders_no_exact_dupes,PASS
3,week_columns_detected,PASS
4,no_artificial_nans_from_coercion,PASS
5,wide_to_long_row_math,PASS
6,grain_unique_in_long_tables,PASS
7,zone_master_is_union,PASS
8,coverage_mismatch_is_real,PASS
9,processed_outputs_coherent,PASS



Overall: PASS (10/10 passed)


## 8) Exportar reporte de validación

Persiste el resultado de validación en `reports/reto1/` para referencia sin necesidad de re-ejecutar el pipeline.

In [10]:
validation_summary = {
    'overall_status': 'PASS' if not failed else 'FAIL',
    'checks_passed': len(checks) - len(failed),
    'checks_failed': len(failed),
    'checks': checks,
}

report_dir = PATHS.reports_dir / 'reto1'
report_dir.mkdir(parents=True, exist_ok=True)

(report_dir / 'pipeline_validation.json').write_text(
    json.dumps(validation_summary, indent=2, ensure_ascii=False), encoding='utf-8'
)

# Markdown summary
md_lines = [
    '# Pipeline Validation Report',
    '',
    f'- Overall status: **{validation_summary["overall_status"]}**',
    f'- Checks passed: {validation_summary["checks_passed"]}',
    f'- Checks failed: {validation_summary["checks_failed"]}',
    '',
    '| Check | Status |',
    '|---|---|',
]
for c in checks:
    md_lines.append(f'| {c["id"]} | **{c["status"]}** |')

(report_dir / 'pipeline_validation.md').write_text('\n'.join(md_lines), encoding='utf-8')

print('Reportes guardados en:', report_dir)
print('  pipeline_validation.json')
print('  pipeline_validation.md')

Reportes guardados en: /home/thechieft/projects/IAeng/reports/reto1
  pipeline_validation.json
  pipeline_validation.md


## 9) Resumen de artefactos generados

Inventario final de outputs producidos por este notebook.

In [11]:
artifacts = [
    ('data/interim', 'metrics_raw_cleaned.parquet', metrics_clean.shape,
     'Métricas limpias (wide). Entrada para métricas long.'),
    ('data/interim', 'orders_raw_cleaned.parquet', orders_clean.shape,
     'Órdenes limpias (wide). Entrada para órdenes long.'),
    ('data/processed', 'metrics_long.parquet', metrics_long.shape,
     'Métricas en formato long. Grano: (COUNTRY, CITY, ZONE, METRIC, WEEK_OFFSET).'),
    ('data/processed', 'orders_long.parquet', orders_long.shape,
     'Órdenes en formato long. Mismo grano.'),
    ('data/processed', 'zone_master.parquet', zone_master.shape,
     'Cobertura de zonas (COVERAGE_CLASS: BOTH / ONLY_METRICS / ONLY_ORDERS).'),
]

artifacts_df = pd.DataFrame(artifacts, columns=['dir', 'file', 'shape', 'description'])
display(artifacts_df)

print('\nPipeline completado. Los notebooks 10, 20, 30, 40 consumen data/processed/')

,dir,file,shape,description
0,data/interim,metrics_raw_cleaned.parquet,"(11610, 32)",Métricas limpias (wide). Entrada para métricas...
1,data/interim,orders_raw_cleaned.parquet,"(1242, 19)",Órdenes limpias (wide). Entrada para órdenes l...
2,data/processed,metrics_long.parquet,"(104490, 14)","Métricas en formato long. Grano: (COUNTRY, CIT..."
3,data/processed,orders_long.parquet,"(11178, 11)",Órdenes en formato long. Mismo grano.
4,data/processed,zone_master.parquet,"(1244, 13)",Cobertura de zonas (COVERAGE_CLASS: BOTH / ONL...



Pipeline completado. Los notebooks 10, 20, 30, 40 consumen data/processed/


---

## Decisiones técnicas documentadas

Estas decisiones afectan el comportamiento del pipeline. Se documentan aquí para que sean auditables sin necesidad de leer el código fuente de `src/`.

### D1. Parquet-only en interim/ y processed/
CSV es opcional y no se persiste por defecto. Razón: menor tamaño, mayor fidelidad de tipos, lectura más rápida.

### D2. Dedup exacto conservador
Solo se eliminan filas idénticas en **todas** las columnas fuente. Duplicados lógicos (misma key, diferente valor) se reportan pero no se colapsan automáticamente. Razón: evitar sobre-limpieza sin validación de negocio.

### D3. Sin imputación de faltantes
Los nulos en columnas de valor semanal se preservan. Se propaga `has_missing_history` como flag. Razón: la imputación debe ser una decisión analítica, no de pipeline base.

### D4. Sin fechas calendario
Solo se usan offsets relativos (`L8W`–`L0W`). Razón: el dato fuente no incluye fechas absolutas y asignarlas requeriría una referencia externa no disponible.

### D5. Trazabilidad completa
`_SOURCE_ROW_NUMBER` permite rastrear cualquier fila en el artefacto procesado hasta su fila original en el Excel. `*_ORIGINAL` preserva el valor textual antes de normalización.

### D6. Mismatch de cobertura es estructural, no un error
264 zonas en órdenes no están en métricas. El `zone_master` lo hace explícito. Los análisis posteriores deben declarar cómo tratan estas zonas.